In [ ]:
from gen_umap_emb import GenerateUmapEmb
from gen_hdb_clus import GenerateHDBSCAN
from pathlib import Path
import pandas as pd
import sys
from forecasting_tools import prophet_forecast, plot_prophet_forecasts
import numpy as np
from gen_hdb_clus import GenerateHDBSCAN

ROOT = Path('/Users/josh/Library/CloudStorage/GoogleDrive-jeverer@gmail.com/My Drive/London Sport/london_sport2')

df_path = ROOT / 'exploration' / 'data' / 'master_data' / '2016_to_2023_master_clustering_data_set.csv'
df = pd.read_csv(df_path)

sys.path.append(str(ROOT / 'src' / 'stream_1' / 'forecasting'))
from forecasting_tools import prophet_forecast, plot_prophet_forecasts

/opt/anaconda3/envs/london/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
df.shape

(117679, 62)

In [3]:
df.columns

Index(['serial', 'year', 'wt_final', 'month', 'LCA_Class', 'LA_2023', 'Age9',
       'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10', 'WorkStat8',
       'Child4', 'HHLiv9', 'Motivation_PC_Q', 'motivd_POP', 'nadult', 'nchild',
       'health', 'comm1', 'anxious', 'happy', 'lifesat', 'lone', 'DVBMI',
       'FruitVegPor', 'READYAB1_POP', 'CULFRQ_1_9_POP', 'VolAny', 'VolCnt',
       'VolDur', 'VolFrqB_Pop', 'volint1', 'volint2', 'volint3', 'volint4',
       'volint5', 'volint6', 'volint7', 'MEMS7_ALL', 'MEMS7_SPORTCOUNT_A01',
       'MEMS7_IN_SPORTCOUNT_A01', 'MEMS7_OUT_SPORTCOUNT_A01',
       'MEMS7_FITNESS_B06', 'MEMS7_WALKALL_C01', 'MEMS7_CYCALL_C02',
       'MEMS7_ACTTRAV_C03', 'MEMS7_DANCEALL_C04', 'MEMS7_TEAMSPORT_C05',
       'MEMS7_RACKETSPORT_C06', 'MEMS7_ADVWATERSPORT_C07', 'MEMS7_LEISURE_C08',
       'MEMS7_COMBATTARGET_C09', 'MEMS7_WINTER_C10', 'MEMS7_RUNATHMULTI_C11',
       'ACT7GR_ALL', 'ACT7GR_SPORTCOUNT_A01', 'Number_Activities',
       'CLUB_SPORTCOUNT_A01', '

In [4]:
df_umap = df.rename(columns={
    'Disab2_POP': 'Disab3',
    'WorkStat8': 'WorkStat10',
    'HHLiv9': 'HHLiv12',
    'Motivation_PC_Q': 'Motiva_POP'
})

cluster_cols = ['Age9', 'NSSEC5', 'Educ6', 'IMD10', 'nchild', 'Motiva_POP', 'motivd_POP',
                'Gend3', 'Disab3', 'Eth7', 'WorkStat10', 'HHLiv12']

df_umap = df_umap[cluster_cols + ['MEMS7_ALL', 'year', 'month', 'LCA_Class', 'LA_2023']].dropna().copy()

In [5]:
umap_gen = GenerateUmapEmb(df_umap)

In [ ]:
emb = umap_gen.fit_umap(num_dimensions=2)
np.save('embeddings/umap_embedding.npy', emb.embedding_)


Categorical Columns for UMAP:
 ['Gend3', 'Disab3', 'Eth7', 'WorkStat10', 'HHLiv12']
Continuous Columns for UMAP
['Age9', 'NSSEC5', 'Educ6', 'IMD10', 'nchild', 'Motiva_POP', 'motivd_POP']
Continuous Shape: (80865, 7)
Categorical Shape: (80865, 29)
Finished fitting continuous....
Finished fitting categorical....
Finished computing intersection...
UMAP fit complete!


In [ ]:

import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=1000, min_samples=25, metric='euclidean').fit(emb.embedding_)
print(len(set(clusterer.labels_)) - 1, 'clusters')
print(f'Noise: {(clusterer.labels_ == -1).mean()*100:.1f}%')

#clusterer = GenerateHDBSCAN(emb.embedding_).hdb()
np.save('embeddings/cluster_labels.npy', clusterer.labels_)

100%|██████████| 11/11 [04:23<00:00, 23.94s/it]


     min_sample  min_cluster cluster_selection     metric  num_clusters  \
1             5           50               eom  manhattan           697   
43           10           50              leaf  manhattan           734   
3             5           50              leaf  manhattan           753   
0             5           50               eom  euclidean           698   
120          30           50               eom  euclidean           658   

     unclustered_prop      dbcv  
1            0.125740  0.360601  
43           0.175379  0.350099  
3            0.165560  0.348502  
0            0.126000  0.347809  
120          0.147097  0.345389  


ValueError: Unable to parse string "2016/17" at position 0

In [ ]:
# embedding = np.load('embeddings/umap_embedding.npy')
# labels = np.load('embeddings/cluster_labels.npy')
# df_umap['cluster'] = labels

df_umap['cluster'] = clusterer.labels_
df_clustered = df_umap[df_umap['cluster'] != -1]

df_clustered['year'] = df_clustered['year'].str.split('/').str[1].astype(int) + 2000
df_cluster_annual = df_clustered.groupby(['cluster', 'year']).agg(
    MEMS7_ALL=('MEMS7_ALL', 'mean')
).reset_index()

In [ ]:
cluster_counts = df_cluster_annual.groupby('cluster').size()
valid_clusters = cluster_counts[cluster_counts >= 7].index
df_cluster_annual = df_cluster_annual[df_cluster_annual['cluster'].isin(valid_clusters)]
print(f'{len(valid_clusters)} clusters with full data')

results = prophet_forecast(df_cluster_annual, t_train_cutoff=5, forecast_steps=2, group_col='cluster')
print(np.mean(results['mape']))


20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing


685 clusters with full data


20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:58 - cmdstanpy - INFO - Chain [1] start processing
20:34:58 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:59 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:59 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:59 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:59 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] start processing
20:34:59 - cmdstanpy - INFO - Chain [1] done processing
20:34:59 - cmdstanpy - INFO - Chain [1] 

56.34989821066454


In [12]:
print(df_cluster_annual.groupby('cluster').size().value_counts().sort_index())

7    685
Name: count, dtype: int64


In [ ]:
import numpy as np

# After UMAP
np.save('embeddings/umap_embedding.npy', emb.embedding_)

# After HDBSCAN
np.save('embeddings/cluster_labels.npy', clusterer.labels_)